In [1]:
import os.path

import numpy as np
from tqdm import tqdm

from alias import *
from utils import *

/opt/homebrew/lib/python3.9/site-packages/statsmodels/tools/_testing.py:19: FutureWarning: pandas.util.testing is deprecated. Use the functions in the public API at pandas.testing instead.
  import pandas.util.testing as tm


In [2]:
dfs = []
dgp_factory_class_list = [
    dgp.LinearFactory,
    dgp.QuickBlockFactory,
    dgp.LinearDriftFactory,
    dgp.LinearSeasonFactory,
    dgp.QuadraticFactory,
    dgp.CubicFactory,
    dgp.SinusoidalFactory,
]

qs = np.round(np.logspace(np.log10(0.01), np.log10(0.499), base=10, num=20), 4)

NUM_ITERS = 1000

for q in qs:
    print(f"q: {q}", flush=True)
    plan = make_plan(
        [
            ("Complete", bal.Complete, est.DifferenceInMeans, {"q": [1 - q, q]}),
            ("Simple", bal.Simple, est.DifferenceInMeans, {"q": [1 - q, q]}),
            # ('Simple + OLS', bal.Simple, est.CovariateAdjustedMean, {'q': [1-q, q]}),
            (
                "QuickBlock",
                bal.QuickBlock,
                est.DifferenceInMeans,
                {"q": q, "k": int(round(1 / q))},
            ),
            ("Smith", bal.Smith, est.DifferenceInMeans, {"q": [1 - q, q]}),
            ("BWD(1)", bal.BWD, est.DifferenceInMeans, {"q": q}),
            ("BWD(0.5)", bal.BWD, est.DifferenceInMeans, {"q": q, "phi": 0.5}),
            # ('BWD + OLS', bal.TBD, est.CovariateAdjustedMean, {'q': q}),
            ("DM", bal.DM, est.DifferenceInMeans, {"q": q}),
            # ('DM + OLS', bal.DM, est.CovariateAdjustedMean, {'q': q}),
        ]
    )
    dgp_factory_list = [factory(N=1000) for factory in dgp_factory_class_list]
    for dgp_factory in dgp_factory_list:
        dgp_name = type(dgp_factory.create_dgp()).__name__
        print(f"\nDGP name: {dgp_name}", flush=True)
        for it in tqdm(range(NUM_ITERS)):
            filename = f"q-results/{dgp_name}_q{int(q * 10000)}_i{it}.csv.gz"
            if not os.path.isfile(filename):
                result = plan.execute(dgp_factory, seed=it * 1001)
                result["iteration"] = it
                result["sample_size"] = q
                result["dgp"] = dgp_name
                result.to_csv(filename, index=False)
                # upload_file(filename, BUCKET)
                dfs.append(result)

q: 0.01

DGP name: LinearDGP


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [01:05<00:00, 15.27it/s]


DGP name: QuickBlockDGP



100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:50<00:00, 19.86it/s]


DGP name: LinearDriftDGP


...


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:38<00:00, 26.10it/s]


DGP name: CubicDGP



100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:37<00:00, 26.90it/s]


DGP name: SinusoidalDGP



100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:39<00:00, 25.54it/s]
